# 05 - PyCaret Workflow vs Manual Workflow

## Why PyCaret exists
PyCaret accelerates experimentation with a compact API and built-in model comparison.

## Tradeoff
- Faster experimentation and less boilerplate
- Lower transparency and lower control than explicit sklearn pipelines


## Definition
PyCaret accelerates model comparison while manual pipelines retain full control and transparency.

## Theory
This section explains the statistical or ML theory behind the technique and why it is useful in credit recovery operations.

## Mathematical Intuition
We translate the idea into score/probability/ranking logic so each metric can be interpreted quantitatively.

## Financial Intuition
We connect the method to borrower affordability, delinquency severity, collateral protection, and expected recoverable cashflows.

## Business Impact
We explain what this enables for collection managers, risk teams, and executives.

## Real-World Example
Teams often prototype in PyCaret, then harden the selected model family in explicit sklearn code.

## Visual Explanation
Charts in this notebook show how model/segment behavior changes across borrower groups.

## Code Explanation
Every code cell below is paired with interpretation so beginners can connect implementation details to business outcomes.

## Interpretation of Results
We summarize what changed, why it matters, and how to act on it.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.loan_recovery import (
    DATA_PATH,
    FIGURES_DIR,
    MODELS_DIR,
    TABLES_DIR,
    REPORTS_DIR,
    LoanDataLoader,
    FeatureEngineer,
    LoanEDA,
    BorrowerSegmenter,
    ModelTrainer,
    ModelEvaluator,
    RecoveryStrategyEngine,
    ModelExplainer,
    DashboardBuilder,
    LazyPredictBenchmark,
    PyCaretWorkflow,
    FLAMLOptimizer,
    SmartLoanRecoveryPipeline,
    load_model,
)

sns.set_theme(style="whitegrid")


In [2]:
def ensure_pipeline_artifacts() -> None:
    required = [
        TABLES_DIR / "manual_model_leaderboard.csv",
        TABLES_DIR / "feature_enriched_data.csv",
        MODELS_DIR / "best_manual_model.joblib",
    ]
    if not all(path.exists() for path in required):
        pipeline = SmartLoanRecoveryPipeline(data_path=DATA_PATH, random_state=42)
        pipeline.run()

ensure_pipeline_artifacts()


In [3]:
enriched = pd.read_csv(TABLES_DIR / "feature_enriched_data.csv")
manual = pd.read_csv(TABLES_DIR / "manual_model_leaderboard.csv")

pycaret = PyCaretWorkflow(random_state=42)
py_out = pycaret.run(enriched.drop(columns=["Borrower_ID"]), target_col="Recovery_Status")

display(py_out.comparison_table)
display(manual)


,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,model_id
0,ExtraTreesClassifier,0.424,0.5049,0.424,0.3460,0.3810,0.0260,0.0273,et
1,RandomForestClassifier,0.392,0.4615,0.392,0.3237,0.3541,-0.0230,-0.0242,rf
2,LogisticRegression,0.384,0.4945,0.384,0.3198,0.3488,-0.0341,-0.0355,lr
3,XGBClassifier,0.344,0.4658,0.344,0.3175,0.3271,-0.0744,-0.0757,xgboost


,model,accuracy,precision_weighted,recall_weighted,f1_weighted,roc_auc_ovr,pr_auc_high_risk
0,SVM,0.44,0.4325,0.44,0.4352,0.5159,0.2518
1,Logistic Regression,0.43,0.4216,0.43,0.4240,0.5445,0.2126
2,XGBoost,0.42,0.3466,0.42,0.3795,0.4770,0.1871
3,CatBoost,0.41,0.3345,0.41,0.3680,0.5004,0.2139
4,LightGBM,0.37,0.3596,0.37,0.3626,0.4607,0.1759
5,Random Forest,0.40,0.3264,0.40,0.3594,0.4675,0.1707
6,Extra Trees,0.39,0.3220,0.39,0.3527,0.4736,0.1713
7,AdaBoost,0.37,0.3139,0.37,0.3371,0.4967,0.1999
8,Gradient Boosting,0.34,0.3169,0.34,0.3260,0.4925,0.1988
9,KNN,0.35,0.3459,0.35,0.3254,0.4944,0.2279


In [4]:
manual_best = manual.iloc[0]
pycaret_best = py_out.comparison_table.iloc[0]
comparison = pd.DataFrame(
    [
        {
            "workflow": "Manual",
            "model": manual_best["model"],
            "accuracy": manual_best["accuracy"],
            "f1": manual_best["f1_weighted"],
            "roc_auc": manual_best["roc_auc_ovr"],
        },
        {
            "workflow": "PyCaret",
            "model": pycaret_best["Model"],
            "accuracy": pycaret_best["Accuracy"],
            "f1": pycaret_best["F1"],
            "roc_auc": pycaret_best["AUC"],
        },
    ]
)
display(comparison)


,workflow,model,accuracy,f1,roc_auc
0,Manual,SVM,0.440,0.4352,0.5159
1,PyCaret,ExtraTreesClassifier,0.424,0.3810,0.5049


## Interpretation
PyCaret should be treated as a strong accelerator for discovery, then hardened with explicit pipelines for production.
